# 🎵 Laboratorio 4: Probabilidad y Estadística con Spotify
**Universidad Rafael Landívar — Curso de Inteligencia Artificial**

---

## Introducción

En este laboratorio trabajaremos con el **Spotify Tracks Dataset**, un conjunto de datos con más de **114,000 canciones** de 125 géneros musicales distintos, recopiladas a través de la API oficial de Spotify.

Cada canción está descrita por características de audio calculadas por los algoritmos de Spotify:

| Columna | Descripción |
|---|---|
| `popularity` | Popularidad de 0 a 100 (basada en número de reproducciones recientes) |
| `danceability` | Qué tan bailable es la canción (0.0 = menos bailable, 1.0 = más bailable) |
| `energy` | Intensidad y actividad percibida (0.0 a 1.0) |
| `valence` | Positividad musical (0.0 = triste/tensa, 1.0 = alegre/eufórica) |
| `tempo` | Tempo estimado en BPM |
| `loudness` | Volumen promedio en dB |
| `acousticness` | Confianza de que la canción es acústica (0.0 a 1.0) |
| `instrumentalness` | Predicción de ausencia de vocales (0.0 a 1.0) |
| `speechiness` | Presencia de palabras habladas (0.0 a 1.0) |
| `liveness` | Probabilidad de que fue grabada en vivo |
| `explicit` | Si contiene lenguaje explícito (True/False) |
| `track_genre` | Género musical de la canción |
| `duration_ms` | Duración en milisegundos |

**Fuente:** [Hugging Face – maharshipandya/spotify-tracks-dataset](https://huggingface.co/datasets/maharshipandya/spotify-tracks-dataset)

In [ ]:
# Importaciones
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Importar funciones del archivo externo
import tarea

# Configuración visual
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='whitegrid', palette='muted')

print('Librerías cargadas correctamente ✓')

---
## Sección 1 — Carga y Preprocesamiento de Datos

El primer paso en cualquier análisis de datos es asegurarnos de que los datos estén limpios y listos para ser procesados. Usaremos las funciones `cargar_datos()` y `preprocesar_datos()` de `tarea.py`.

In [ ]:
# Carga del dataset original
df_raw = tarea.cargar_datos()
print(f'Dataset original: {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas')
df_raw.head(3)

In [ ]:
# Resumen de valores nulos antes del preprocesamiento
nulos = df_raw.isnull().sum()
print('Valores nulos por columna (solo columnas con nulos):')
print(nulos[nulos > 0])

In [ ]:
# Aplicar preprocesamiento
df = tarea.preprocesar_datos(df_raw)
print(f'Dataset preprocesado: {df.shape[0]:,} filas × {df.shape[1]} columnas')
print(f'Filas eliminadas: {df_raw.shape[0] - df.shape[0]:,}')
df.dtypes

---
## Sección 2 — Análisis Exploratorio de Datos (EDA)

Antes de calcular probabilidades, exploramos la estructura y distribución general del dataset.

In [ ]:
# Estadísticas descriptivas de las variables numéricas principales
columnas_numericas = ['popularity', 'danceability', 'energy', 'valence', 'tempo', 'loudness']
df[columnas_numericas].describe().round(3)

In [ ]:
# ── Gráfico 1: Distribución de Popularidad ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma
axes[0].hist(df['popularity'], bins=50, color='#1DB954', edgecolor='white', linewidth=0.5)
axes[0].axvline(df['popularity'].mean(), color='black', linestyle='--', linewidth=1.5,
                label=f"Media: {df['popularity'].mean():.1f}")
axes[0].axvline(df['popularity'].median(), color='gray', linestyle=':', linewidth=1.5,
                label=f"Mediana: {df['popularity'].median():.1f}")
axes[0].set_title('Distribución de Popularidad', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Popularidad (0–100)')
axes[0].set_ylabel('Cantidad de canciones')
axes[0].legend()

# Boxplot
axes[1].boxplot(df['popularity'], vert=False, patch_artist=True,
                boxprops=dict(facecolor='#1DB954', alpha=0.7))
axes[1].set_title('Boxplot de Popularidad', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Popularidad (0–100)')
axes[1].set_yticks([])

plt.suptitle('Figura 1 — Análisis de la variable Popularidad', fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Gráfico 2: Danceability vs Energy por género (muestra) ─────────────────
# Tomamos los 6 géneros con más canciones
top6_generos = df['track_genre'].value_counts().head(6).index.tolist()
df_top6 = df[df['track_genre'].isin(top6_generos)]

paleta = sns.color_palette('tab10', n_colors=6)
fig, ax = plt.subplots(figsize=(10, 6))

for i, genero in enumerate(top6_generos):
    subset = df_top6[df_top6['track_genre'] == genero].sample(min(300, len(df_top6)), random_state=42)
    ax.scatter(subset['danceability'], subset['energy'],
               alpha=0.35, s=20, color=paleta[i], label=genero)

ax.set_xlabel('Danceability', fontsize=12)
ax.set_ylabel('Energy', fontsize=12)
ax.set_title('Figura 2 — Danceability vs Energy por Género Musical\n(muestra de los 6 géneros con más canciones)',
             fontsize=12, fontweight='bold')
ax.legend(title='Género', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# ── Gráfico 3: Top 10 géneros por popularidad promedio ─────────────────────
top10 = tarea.top_n_por_metrica(df, 'track_genre', 'popularity', n=10, funcion_agregacion='mean')

fig, ax = plt.subplots(figsize=(10, 5))
colores = plt.cm.RdYlGn(np.linspace(0.3, 0.9, 10))
bars = ax.barh(top10['track_genre'][::-1], top10['popularity'][::-1],
               color=colores, edgecolor='white')

for bar, val in zip(bars, top10['popularity'][::-1]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}', va='center', fontsize=9)

ax.set_xlabel('Popularidad Promedio', fontsize=11)
ax.set_title('Figura 3 — Top 10 Géneros por Popularidad Promedio', fontsize=12, fontweight='bold')
ax.set_xlim(0, top10['popularity'].max() + 5)
plt.tight_layout()
plt.show()

---
## Sección 3 — Medidas de Tendencia Central y Dispersión

Aplicamos las funciones de `tarea.py` para calcular estadísticas descriptivas sobre las variables más importantes del dataset.

In [ ]:
# Tendencia central para múltiples variables
variables = ['popularity', 'danceability', 'energy', 'tempo', 'valence']

print(f'{"Variable":<20} {"Media":>10} {"Mediana":>10} {"Moda":>10}')
print('-' * 52)
for var in variables:
    res = tarea.calcular_medidas_tendencia_central(df, var)
    print(f'{var:<20} {res["media"]:>10.3f} {res["mediana"]:>10.3f} {res["moda"]:>10.3f}')

In [ ]:
# Dispersión para múltiples variables
print(f'{"Variable":<20} {"Varianza":>15} {"Desv. Estándar":>16}')
print('-' * 53)
for var in variables:
    res = tarea.calcular_medidas_dispersion(df, var)
    print(f'{var:<20} {res["varianza"]:>15.4f} {res["desviacion_estandar"]:>16.4f}')

---
## Sección 4 — Cálculo de Probabilidades

### 4.1 Probabilidades Totales (marginales)

In [ ]:
# Probabilidad de que una canción sea explícita
p_explicit = tarea.calcular_probabilidad_total(df, 'explicit', True)
print(f'P(explicit = True)  = {p_explicit:.4f}  ({p_explicit*100:.2f}%)')

# Probabilidad de que una canción sea del género 'pop'
p_pop = tarea.calcular_probabilidad_total(df, 'track_genre', 'pop')
print(f'P(género = pop)     = {p_pop:.4f}  ({p_pop*100:.2f}%)')

# Probabilidad de que una canción sea del género 'rock'
p_rock = tarea.calcular_probabilidad_total(df, 'track_genre', 'rock')
print(f'P(género = rock)    = {p_rock:.4f}  ({p_rock*100:.2f}%)')

In [ ]:
# Clasificamos popularidad para usar en probabilidades
df = tarea.clasificar_popularidad(df)

# Probabilidad de cada categoría de popularidad
for cat in ['Baja', 'Media', 'Alta']:
    p = tarea.calcular_probabilidad_total(df, 'categoria_popularidad', cat)
    print(f'P(popularidad = {cat:<6}) = {p:.4f}  ({p*100:.2f}%)')

### 4.2 Probabilidades Condicionales

La **probabilidad condicional** P(A | B) nos permite responder preguntas como: *"dado que una canción es de cierto género, ¿qué tan probable es que sea popular?"*

In [ ]:
# P(popular = Alta | género = pop)
generos_analisis = ['pop', 'rock', 'hip-hop', 'jazz', 'classical', 'edm']
print('P(popularidad = Alta | género = X)\n')

resultados_cond = {}
for genero in generos_analisis:
    p = tarea.calcular_probabilidad_condicional(
        df, 'track_genre', genero, 'categoria_popularidad', 'Alta'
    )
    resultados_cond[genero] = p
    print(f'  P(Alta | {genero:<12}) = {p:.4f}  ({p*100:.2f}%)')

In [ ]:
# P(explicit = True | popularidad = Alta)
p_exp_dado_alta = tarea.calcular_probabilidad_condicional(
    df, 'categoria_popularidad', 'Alta', 'explicit', True
)
p_exp_dado_baja = tarea.calcular_probabilidad_condicional(
    df, 'categoria_popularidad', 'Baja', 'explicit', True
)
print(f'P(explicit=True | popularidad=Alta) = {p_exp_dado_alta:.4f}  ({p_exp_dado_alta*100:.2f}%)')
print(f'P(explicit=True | popularidad=Baja) = {p_exp_dado_baja:.4f}  ({p_exp_dado_baja*100:.2f}%)')
print()
print('¿El contenido explícito está asociado a mayor popularidad?')
if p_exp_dado_alta > p_exp_dado_baja:
    print('  ✓ Sí: las canciones populares tienen mayor probabilidad de ser explícitas.')
else:
    print('  ✗ No: las canciones populares no tienen más contenido explícito.')

---
## Sección 5 — Análisis de Correlación

Calculamos la correlación de Pearson entre las características de audio para identificar relaciones lineales.

In [ ]:
# Correlaciones con popularidad
features = ['danceability', 'energy', 'valence', 'tempo', 'loudness', 'acousticness']
print('Correlación de Pearson con Popularidad:\n')
for feat in features:
    corr = tarea.calcular_correlacion(df, 'popularity', feat)
    barra = '█' * int(abs(corr) * 20)
    direccion = '+' if corr >= 0 else '-'
    print(f'  {feat:<18} {direccion}{barra:<20} {corr:+.4f}')

In [ ]:
# ── Gráfico 4: Mapa de calor de correlaciones ───────────────────────────────
cols_corr = ['popularity', 'danceability', 'energy', 'valence',
             'tempo', 'loudness', 'acousticness', 'speechiness']
matriz_corr = df[cols_corr].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(matriz_corr, dtype=bool))
sns.heatmap(
    matriz_corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5, ax=ax,
    cbar_kws={'shrink': 0.8}
)
ax.set_title('Figura 4 — Matriz de Correlación de Características de Audio\n(triángulo inferior)',
             fontsize=12, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

---
## Sección 6 — Distribuciones por Categoría

Usamos `distribucion_por_categoria()` para explorar la composición del dataset.

In [ ]:
# Top 15 géneros por número de canciones
dist_genero = tarea.distribucion_por_categoria(df, 'track_genre')
print('Top 15 géneros (por proporción):')
print(dist_genero.head(15).to_string(index=False))

In [ ]:
# ── Gráfico 5: Distribución de categorías de popularidad ────────────────────
dist_pop = tarea.distribucion_por_categoria(df, 'categoria_popularidad')

# Orden lógico
orden = ['Baja', 'Media', 'Alta']
dist_pop = dist_pop.set_index('categoria_popularidad').reindex(orden).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Barras
colores_cat = ['#E74C3C', '#F39C12', '#2ECC71']
axes[0].bar(dist_pop['categoria_popularidad'], dist_pop['proporcion'],
            color=colores_cat, edgecolor='white', linewidth=1.2)
for i, (_, row) in enumerate(dist_pop.iterrows()):
    axes[0].text(i, row['proporcion'] + 0.005, f"{row['proporcion']*100:.1f}%",
                 ha='center', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Proporción')
axes[0].set_title('Distribución por Categoría de Popularidad', fontweight='bold')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))

# Pie
axes[1].pie(dist_pop['proporcion'], labels=dist_pop['categoria_popularidad'],
            autopct='%1.1f%%', colors=colores_cat, startangle=140,
            wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Proporción de Categorías de Popularidad', fontweight='bold')

plt.suptitle('Figura 5 — Distribución de Categorías de Popularidad', fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

---
## Sección 7 — Actividad Guiada: Tu Propio Análisis

Responde las siguientes preguntas utilizando las funciones de `tarea.py`. Documenta tus hallazgos con celdas de Markdown.

### Pregunta 1
¿Cuál es la probabilidad de que una canción sea del género `'acoustic'`? ¿Y dado que es acústica (`acousticness > 0.7`), qué tan probable es que sea de popularidad Alta?

In [ ]:
# Escribe tu código aquí


### Pregunta 2
Calcula la media, mediana y desviación estándar de `danceability` **solo para canciones de popularidad Alta**. ¿Difieren significativamente respecto al dataset completo?

In [ ]:
# Escribe tu código aquí


### Pregunta 3
Usa `top_n_por_metrica()` para encontrar los 5 géneros con mayor `valence` (positividad musical) promedio. ¿Te sorprende algún resultado? Justifica con una celda de Markdown.

In [ ]:
# Escribe tu código aquí


### Pregunta 4 (Análisis libre)
Elige **dos variables numéricas** del dataset que creas que podrían estar relacionadas con la popularidad. Calcula su correlación con `popularity`, genera un gráfico de dispersión y argumenta tus conclusiones.

In [ ]:
# Escribe tu código aquí


---
## Sección 8 — Conclusiones

*Escribe aquí tus conclusiones del laboratorio. Responde al menos las siguientes preguntas:*

1. ¿Qué característica de audio tiene mayor correlación con la popularidad?
2. ¿El contenido explícito influye en la popularidad de una canción?
3. ¿Qué género musical tiene mayor probabilidad de producir canciones populares?
4. ¿Qué aprendiste sobre probabilidad condicional a partir de este análisis?

> **Reemplaza este bloque con tus propias conclusiones.**